# PKG TM customer x month (TM account analysis + customer fact)

Builds two monthly tables in `bdahd01p_dlcdi1_cdi_tm`, keyed by power id and by **analysis month** (same month meaning as PKG):

- `pkg_tm_cust_month`: one row per power id per month. TM account analysis block (accounts, opened / closed / gone, balances, analyzed charges, waivers, earnings credit, billed, deposit FTP value) and customer fact block (PD grade now / 1 / 3 / 12 months ago, default flag, exposures, past due, 12-month revenue, economic profit, industry risk, exposure type).
- `pkg_tm_cust_prod_month`: one row per power id x TM product (`prod_desc_ofs`) per month: service codes, accounts, rows, rows with volume, price x volume revenue, first month seen.

Rules (checked 2026-11-08): OFS snapshot month M holds the analysis of month M-1; account key = bank + appl_id + src_acct_no; acct_dim_key changes every month, so revenue rows reach their customer through the balance table of the same snapshot (own power id as fallback); when under 80% of keyed revenue rows find their account there (seen for 2023-08), the previous / next balance snapshot, the current or period customer key (curnt_cust_dim_key, cust_dim_key) or the customer dimension is used instead, the first that reaches 80%; when none does (2023-08: every surrogate key in that revenue load is from another key generation), the revenue rows' own power id is used and the month is flagged as partial. The run log records the method; customer fact = last as_of_dt of the month, its latest FULL load (largest time_key among loads with at least half the rows of the date's largest load). Month-end dates carry two full loads (day of and next day; the second sometimes regenerates cust_dim_keys, which double-counted DHE under the earlier rule) plus small reloads that carry no DHE. OFS amounts (balances, charges, billed, earnings credit, FTP) are summed over leaf accounts only: parent rows of the billing hierarchy have no power id or customer key and repeat their children's amounts (checked 2026-11-09). Customers: everyone with TM analysis accounts this month or last month, plus PKG TM customers (pkg_cust_dim) found in the customer fact. Power ids missing from all sources are left out.

Cells
1. Setup and config
2. Inputs, month plan and resume status (a month is done only when both tables carry a marker with the current configuration signature and the previous month's build id)
3. Checks on the latest month: revenue-to-balance account link, revenue reconciliation by system, product row growth
4. Build: customer fact staged once for the months to build, then month by month with exact reconciliation; prints progress
5. Register both tables, monthly summary, column dictionary

After the run, in Impala: INVALIDATE METADATA for both tables.

In [ ]:
# ============================================================
# 1. Setup and config
# ============================================================
import os, re, html, warnings, datetime as _dt
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

# --- NumPy alias shim (old pyspark code paths use removed aliases) ---
import numpy as np
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp),
               ("bool", bool), ("object", object)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)

# --- pandas shim: PySpark 3.3 createDataFrame calls iteritems (removed in pandas 2.0) ---
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("pkg_tm_cust_month")
         .config("spark.ui.showConsoleProgress", "false")   # no stage progress bars in cell outputs
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:  # silence Atlas lineage hook + harmless executor-release noise (log4j2 on Spark 3.3)
    _L = spark._jvm.org.apache.logging.log4j
    for n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
              "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
              "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(n, _L.Level.OFF)
except Exception:
    pass
# Unparseable date strings -> NULL instead of SparkUpgradeException (we probe many formats)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------------- config ----------------
DB           = "bdahd01p_dlcdi1_cdi_tm"
T_CUST       = "pkg_tm_cust_month"         # one row per power id per analysis month
T_PROD       = "pkg_tm_cust_prod_month"    # one row per power id x TM product per analysis month
OFS_BAL      = "clhhd01p_clh.cdm_tm_ofs_bal_analysis_fact_v"
OFS_REV      = "clhhd01p_clh.cdm_tm_ofs_revenue_fact_v"
CUST_FACT    = "clhhd01p_clh.cdm_customer_fact_v"
PD_DIM       = "clhhd01p_clh.cdm_probability_default_dim_v"
IRIS_DIM     = "clhhd01p_clh.cdm_iris_dim_v"
EXPSR_DIM    = "clhhd01p_clh.cdm_cust_credit_expsr_type_dim_v"
PKG_DIM      = f"{DB}.pkg_cust_dim"
START_MONTH  = "2023-01"   # first analysis month (12 months of look-back before PKG's 2024-01)
END_MONTH    = "latest"    # or "YYYY-MM"
SHIFT        = 1           # OFS snapshot month = analysis month + 1 (checked 2026-11-08)
REBUILD_FROM = None        # "YYYY-MM": rebuild this month and every later one
RUN_CHECKS   = True        # cell 3 checks on the latest month
MIN_ACCT_LINK = 0.80       # share of revenue rows with an account key that a link method must reach to be used
CUST_DIM     = "clhhd01p_clh.cdm_customer_dim_v"   # optional link fallback (cust_dim_key -> power id), used only if it has both
ALLOW_OWN_ID_ONLY = True
FULL_LOAD_SHARE = 0.5      # customer fact: a load counts as full when it has >= this share of the date's largest load
LEAF_ONLY_AMOUNTS = True   # OFS amounts summed over leaf accounts only (parent rows of the billing hierarchy repeat children)
DHE_JUMP     = 1.5         # cell 5 flags months whose DHE moves more than this factor against both neighbours   # no key links a month's revenue rows: use their own power id (partial month, flagged)
GROWTH_FROM  = "2025-12"   # analysis month compared with the latest in the row-growth check
VERSION      = "tm_cust_month_v3"
OUT_DIR      = "../metrics/tm_cust_month"
os.makedirs(OUT_DIR, exist_ok=True)
spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")   # month 'YYYY-MM' stays text

# ---------------- helpers ----------------
CSS = """<style>
.cx {font-family: Arial, sans-serif; font-size: 12px;}
.cx h3 {margin: 14px 0 4px 0; font-size: 14px;}
.cx h4 {margin: 10px 0 3px 0; font-size: 13px; color: #1f3b63;}
.cx table {border-collapse: collapse; margin: 2px 0 8px 0;}
.cx th {background: #e8eef6; border: 1px solid #c8d2e0; padding: 2px 6px; text-align: left; white-space: nowrap;}
.cx td {border: 1px solid #dde3ec; padding: 2px 6px; vertical-align: top; max-width: 420px; overflow-wrap: anywhere;}
.cx .note {color: #555; margin: 2px 0;}
.cx .warn {color: #a33; font-weight: bold;}
.cx pre {background: #f6f8fa; border: 1px solid #dde3ec; padding: 6px; white-space: pre-wrap; font-size: 11px; max-width: 1400px;}
</style>"""

def esc(x):
    return html.escape("" if x is None else str(x))

def tbl(pdf, max_rows=None):
    """pandas DataFrame -> compact HTML table (values escaped)."""
    if pdf is None or len(pdf) == 0:
        return "<p class='note'>(no rows)</p>"
    if max_rows is not None:
        pdf = pdf.head(max_rows)
    head = "".join(f"<th>{esc(c)}</th>" for c in pdf.columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row) + "</tr>"
                   for row in pdf.astype(object).where(pdf.notna(), "").itertuples(index=False))
    return f"<table><tr>{head}</tr>{body}</table>"

def show(parts):
    """One consolidated HTML output per cell; tex2jax_ignore keeps any dollar sign from turning into math."""
    display(HTML(CSS + "<div class='cx tex2jax_ignore mathjax_ignore'>" + "".join(parts) + "</div>"))

def fnum(x):
    if x is None:
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if np.isnan(x):
        return ""
    a = abs(x)
    for d, s in ((1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{x / d:.2f}{s}"
    if x == int(x):
        return str(int(x))
    return f"{x:.4g}"

def fpct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{100 * float(x):.{digits}f}%"

def qcol(c):
    return F.col(f"`{c}`")

NUMERIC_T = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
COMPLEX_T = (T.ArrayType, T.MapType, T.StructType, T.BinaryType)
DATE_NAME = re.compile(r"(_dt$|_dte$|_date$|^date_|_date_|_ts$|_tmstmp|timestamp|^dt_|_dt_|asof|as_of|snap|_month$|_mo$|_yrmo|yyyymm|period)")

def kind_of(dtype):
    if isinstance(dtype, (T.DateType, T.TimestampType)): return "date"
    if isinstance(dtype, T.BooleanType): return "bool"
    if isinstance(dtype, NUMERIC_T): return "num"
    if isinstance(dtype, COMPLEX_T): return "complex"
    return "str"

def is_date_like(name, dtype):
    k = kind_of(dtype)
    if k == "date":
        return True
    return k in ("str", "num") and bool(DATE_NAME.search(name.lower())) \
        and not isinstance(dtype, (T.FloatType, T.DoubleType))

def date_expr(c, dtype):
    """Parse a date-like column of any common storage format to DATE (NULL if unparseable)."""
    col = qcol(c)
    if isinstance(dtype, T.DateType):
        return col
    if isinstance(dtype, T.TimestampType):
        return F.to_date(col)
    s = F.trim(col.cast("string"))
    return (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
             .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd"))
             .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}"),
                   F.to_date(F.regexp_extract(s, r"^(\d{1,2}/\d{1,2}/\d{4})", 1), "M/d/yyyy"))
             .when(s.rlike(r"^\d{6}$"), F.to_date(F.concat(s, F.lit("01")), "yyyyMMdd"))
             .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd")))

def real_date(p):
    """Drop placeholder dates (year <= 1900 or >= 2100, e.g. 0001-01-01 / 9999-12-31) for min/max."""
    return F.when(F.year(p).between(1901, 2099), p)

show([f"<h3>Setup</h3><p>Spark {esc(spark.version)} · pandas {esc(pd.__version__)} · numpy {esc(np.__version__)}</p>",
      tbl(pd.DataFrame([("Output", f"{DB}.{T_CUST}, {DB}.{T_PROD}"), ("Analysis months", f"{START_MONTH} to {END_MONTH}"),
                        ("SHIFT", SHIFT), ("REBUILD_FROM", REBUILD_FROM), ("RUN_CHECKS", RUN_CHECKS),
                        ("VERSION", VERSION), ("OUT_DIR", os.path.abspath(OUT_DIR))], columns=["setting", "value"]))])


In [ ]:
# ============================================================
# 2. Inputs, month plan and resume status
# ============================================================
import json, hashlib, uuid, time

def ym_int(m):
    return int(m[:4]) * 100 + int(m[5:7])

def ym_str(i):
    return f"{int(i) // 100}-{int(i) % 100:02d}"

def ym_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"

def snap(m):
    """yyyymm of the OFS snapshot that holds analysis month m."""
    return ym_int(ym_add(m, SHIFT))

def clean_pwr(c):
    """Power id as 10-digit text; '-1', blanks, zeros and non-digits -> NULL."""
    s = F.trim(F.col(c).cast("string"))
    s = F.when(s.rlike(r"^[0-9]{1,10}$"), F.lpad(s, 10, "0"))
    return F.when(s != F.lit("0000000000"), s)

def real_dt(c):
    """String / yyyymmdd date -> DATE; placeholders (9999-12-31, 0, blank) -> NULL."""
    s = F.trim(F.col(c).cast("string"))
    d = (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
          .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd")))
    return F.when(F.year(d).between(1901, 2099), d)

def flagv(c):
    return F.upper(F.trim(F.col(c).cast("string")))

def current_rows(full, key):
    """Current row per key of a dimension: active, not purged, open-ended, newest effective / update date."""
    d = spark.table(full)
    cols = d.columns
    if key not in cols:
        raise RuntimeError(f"{full} has no column {key}")
    order = []
    if "active_flg" in cols:
        order.append(F.when(flagv("active_flg").isin("Y", "1", "TRUE", "A"), 0).otherwise(1))
    if "purge_flg" in cols:
        order.append(F.when(flagv("purge_flg").isin("Y", "1", "TRUE"), 1).otherwise(0))
    if "end_dt" in cols:
        order.append(F.col("end_dt").desc_nulls_first())
    for c in ("eff_dt", "update_dt", "create_dt", "src_load_dt"):
        if c in cols:
            order.append(F.col(c).desc_nulls_last())
    w = Window.partitionBy(key).orderBy(*(order or [F.lit(1)]))
    return d.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def by_month(table):
    """{yyyymm: latest time_key in that month} from SHOW PARTITIONS (fallback: distinct time_key)."""
    try:
        pp = spark.sql(f"SHOW PARTITIONS {table}").toPandas().iloc[:, 0].astype(str)
        tks = {int(v.split("time_key=", 1)[1].split("/")[0]) for v in pp if "time_key=" in v}
    except Exception:
        tks = set()
    if not tks:
        tks = {int(r[0]) for r in spark.table(table).select("time_key").distinct().collect() if r[0] is not None}
    out = {}
    for t in tks:
        if 19000101 <= t <= 21001231:
            out[t // 100] = max(out.get(t // 100, 0), t)
    return out

# ---- Hadoop file helpers (markers, staging) ----
_jvm, _hconf = spark._jvm, spark._jsc.hadoopConfiguration()
def _fp(p):
    P = _jvm.org.apache.hadoop.fs.Path(p)
    return P.getFileSystem(_hconf), P
def fs_exists(p):
    fs, P = _fp(p)
    return bool(fs.exists(P))
def fs_write(p, text):
    fs, P = _fp(p)
    out = fs.create(P, True)
    out.write(bytearray(text.encode("utf-8")))
    out.close()
def fs_read(p):
    fs, P = _fp(p)
    if not fs.exists(P):
        return None
    rd = _jvm.java.io.BufferedReader(_jvm.java.io.InputStreamReader(fs.open(P), "UTF-8"))
    lines, line = [], rd.readLine()
    while line is not None:
        lines.append(line)
        line = rd.readLine()
    rd.close()
    return "\n".join(lines)
def fs_delete(p):
    fs, P = _fp(p)
    if fs.exists(P):
        fs.delete(P, True)

# ---- sources and required columns (names from the 2026-11-08 profiles) ----
bal, rev, fact = spark.table(OFS_BAL), spark.table(OFS_REV), spark.table(CUST_FACT)
KEY = ["bank", "appl_id", "src_acct_no"]
BAL_SUMS = {"inv_bal": "avg_investable_bal", "avg_bal": "cor_tda_avg_bal", "analyzed_chrg": "analyzed_chrg",
            "waived": "waived_results", "eca_used": "eca_used", "billed": "tot_bdf_billed",
            "ecr_amt": "earnings_credit_amt", "ftp_value": "cor_tda_ftp_on_avg_bal"}
FACT_SUMS = {"dhe": "dhe", "wrre": "cust_wrre", "mre": "cust_mre", "soft_exposure": "cust_ste",
             "watchlist_exposure": "watchlist_exposure", "gross_rev_12m": "gross_rev_12_mnth",
             "credit_rev_12m": "credit_rev_12_mnth", "non_credit_rev_12m": "non_credit_rev_12_mnth",
             "tm_rev_12m": "tm_rev_12_mnth", "ep_12m": "ep_amt_12_mnth"}
FACT_MAX = {"past_due_days": "cust_past_due_days"}
need = {OFS_BAL: KEY + ["time_key", "seq_id", "acct_dim_key", "curnt_cust_dim_key", "cust_dim_key", "cust_pwr_id", "acct_close_dt", "curnt_mnth_nbr",
                        "earnings_credit_rate", "data_src_cd", "tot_pxv_rev", "tm_pxv_rev", "hier_leaf_node"] + list(BAL_SUMS.values()),
        OFS_REV: ["time_key", "acct_dim_key", "curnt_cust_dim_key", "cust_dim_key", "cust_pwr_id", "prod_desc_ofs", "tm_service_cd_ofs", "volume_ofs",
                  "pxv_ofs", "other_rev_ofs", "memo_rev_ofs", "subsystem_identifier_ofs"],
        CUST_FACT: ["as_of_dt", "time_key", "cust_dim_key", "cust_pwr_id", "pd_key", "pd_cd_1_mnth_key",
                    "pd_cd_3_mnth_key", "pd_cd_12_mnth_key", "dhe", "gross_rev_12_mnth", "tm_rev_12_mnth",
                    "iris_dim_key", "cust_credit_expsr_type_dim_key"]}
for t, df in ((OFS_BAL, bal), (OFS_REV, rev), (CUST_FACT, fact)):
    miss = [c for c in need[t] if c not in df.columns]
    if miss:
        raise RuntimeError(f"{t} lacks columns {miss}")
fact_missing = sorted(s for s in list(FACT_SUMS.values()) + list(FACT_MAX.values()) if s not in fact.columns)

# ---- dimensions ----
pd_cur = current_rows(PD_DIM, "pd_key")
for c in ("pd_cd", "pd_pct"):
    if c not in pd_cur.columns:
        raise RuntimeError(f"{PD_DIM} lacks column {c}")
pdp = pd_cur.select(F.col("pd_key").cast("long").alias("k"), F.trim(F.col("pd_cd").cast("string")).alias("cd"),
                    F.col("pd_pct").cast("double").alias("pct")).toPandas()
if pdp.empty:
    raise RuntimeError(f"{PD_DIM} is empty")
PD_SCALE = 100.0 if pdp["pct"].max() > 1.5 else 1.0     # store PD as a fraction (1.0 = 100% = defaulted)
pdp["pct"] = pdp["pct"] / PD_SCALE
PD_PCT = F.create_map(*[x for k, p in zip(pdp["k"], pdp["pct"])
                        for x in (F.lit(int(k)), F.lit(None if pd.isna(p) else float(p)).cast("double"))])
PD_CD = F.create_map(*[x for k, c in zip(pdp["k"], pdp["cd"])
                       for x in (F.lit(int(k)), F.lit(None if c is None else str(c)).cast("string"))])

iris_cur = current_rows(IRIS_DIM, "iris_dim_key")
IRIS_COLS = {"iris_condition": ("iris_industry_condition_desc", "string"),
             "iris_direction": ("iris_direction_of_risk", "string"), "iris_score": ("iris_score", "double")}
iris_missing = [s for s, _ in IRIS_COLS.values() if s not in iris_cur.columns]
IRIS = iris_cur.select(F.col("iris_dim_key").cast("long").alias("iris_dim_key"),
                       *[(F.col(s).cast(ty) if s in iris_cur.columns else F.lit(None).cast(ty)).alias(d)
                         for d, (s, ty) in IRIS_COLS.items()])

ex_cur = current_rows(EXPSR_DIM, "cust_credit_expsr_type_dim_key")
EX_LAB = next((c for c in ("cust_credit_expsr_type_cd", "cust_credit_expsr_type_desc") if c in ex_cur.columns), None)
if EX_LAB:
    exp_ = ex_cur.select(F.col("cust_credit_expsr_type_dim_key").cast("long").alias("k"),
                         F.trim(F.col(EX_LAB).cast("string")).alias("v")).toPandas()
    EX_MAP = F.create_map(*[x for k, v in zip(exp_["k"], exp_["v"]) for x in (F.lit(int(k)), F.lit(v).cast("string"))])
else:
    EX_MAP = None

pk = (spark.table(PKG_DIM).select(clean_pwr("customer_pwr_id").alias("customer_pwr_id"))
           .filter(F.col("customer_pwr_id").isNotNull()).distinct().persist(StorageLevel.MEMORY_AND_DISK))
N_PK = pk.count()

# ---- snapshots ----
BAL_TK, REV_TK = by_month(OFS_BAL), by_month(OFS_REV)
fd = fact.groupBy("as_of_dt", "time_key").count().toPandas().dropna()
fd["as_of_dt"], fd["time_key"] = fd["as_of_dt"].astype("int64"), fd["time_key"].astype("int64")
fd = fd[(fd["as_of_dt"] >= 19000101) & (fd["as_of_dt"] <= 21001231)]
biggest = fd.groupby("as_of_dt")["count"].max()
full = fd[fd["count"] >= FULL_LOAD_SHARE * fd["as_of_dt"].map(biggest)]
best = full.sort_values("time_key").groupby("as_of_dt").tail(1).set_index("as_of_dt")   # latest full load
nload = fd.groupby("as_of_dt").size()
FACT_SNAP = {}
for a in sorted(best.index):        # later dates overwrite: last as_of_dt of the month
    FACT_SNAP[ym_str(a // 100)] = (int(a), int(best.loc[a, "time_key"]), int(nload[a]), int(best.loc[a, "count"]),
                                   int(biggest[a]))

last_ok = min(ym_add(ym_str(max(BAL_TK)), -SHIFT), ym_add(ym_str(max(REV_TK)), -SHIFT), max(FACT_SNAP))
END = last_ok if END_MONTH == "latest" else END_MONTH
MONTHS, m = [], START_MONTH
while m <= END:
    MONTHS.append(m)
    m = ym_add(m, 1)
gaps = []
for m in MONTHS:
    for what, ok in (("balance", snap(m) in BAL_TK), ("balance (previous month)", snap(ym_add(m, -1)) in BAL_TK),
                     ("revenue", snap(m) in REV_TK), ("customer fact", m in FACT_SNAP)):
        if not ok:
            gaps.append(f"{m}: no {what} snapshot")
if not MONTHS or gaps:
    raise RuntimeError("Month plan incomplete: " + ("; ".join(gaps[:20]) if gaps else "no months"))

# ---- output locations, signature, resume status ----
_d = spark.sql(f"DESCRIBE DATABASE EXTENDED {DB}").toPandas()
LOC = str(_d.loc[_d.iloc[:, 0].astype(str).str.strip().str.lower() == "location"].iloc[0, 1]).rstrip("/")
PATH_CUST, PATH_PROD = f"{LOC}/{T_CUST}", f"{LOC}/{T_PROD}"
STAGE = f"{LOC}/_pkg_tm_staging/cust_fact_month"
STATE = f"{LOC}/_pkg_tm_staging/prod_first_seen"
SIG = hashlib.sha1(json.dumps({"v": VERSION, "shift": SHIFT, "start": START_MONTH, "key": KEY,
                               "src": [OFS_BAL, OFS_REV, CUST_FACT, PD_DIM, IRIS_DIM, EXPSR_DIM, PKG_DIM],
                               "fact_missing": fact_missing, "iris_missing": iris_missing, "ex_lab": EX_LAB,
                               "pd_scale": PD_SCALE}, sort_keys=True).encode()).hexdigest()[:12]

def marker(m):
    a = fs_read(f"{PATH_CUST}/month={m}/_PKG_MARKER.json")
    b = fs_read(f"{PATH_PROD}/month={m}/_PKG_MARKER.json")
    if not a or a != b:
        return None
    try:
        return json.loads(a)
    except ValueError:
        return None

def plan_months(use_rebuild=True):
    """Months to build: the first month without a valid marker chain and everything after it.
    REBUILD_FROM applies only when planning a build (cell 4), not when listing finished months (cell 5)."""
    plan, prev_id, rows = [], None, []
    for m in MONTHS:
        mk = marker(m)
        ok = (not plan and mk is not None and mk.get("sig") == SIG and mk.get("prev") == prev_id
              and (not use_rebuild or REBUILD_FROM is None or m < REBUILD_FROM))
        if ok:
            prev_id = mk["build_id"]
        else:
            plan.append(m)
        rows.append((m, "done" if ok else "to build", prev_id if ok else None))
    return plan, rows

PLAN, status = plan_months()
mt = pd.DataFrame([{"analysis month": m, "balance snapshot": BAL_TK[snap(m)],
                    "previous balance snapshot": BAL_TK[snap(ym_add(m, -1))], "revenue snapshot": REV_TK[snap(m)],
                    "customer fact as_of_dt": FACT_SNAP[m][0], "loads that day": FACT_SNAP[m][2],
                    "load used (time_key)": FACT_SNAP[m][1], "rows in load used": f"{FACT_SNAP[m][3]:,}",
                    "rows in largest load": f"{FACT_SNAP[m][4]:,}",
                    "status": s} for m, s, _ in status])
warn = []
if fact_missing:
    warn.append(f"Customer fact columns not found, left empty: {', '.join(fact_missing)}")
if iris_missing:
    warn.append(f"IRIS columns not found, left empty: {', '.join(iris_missing)}")
if not EX_LAB:
    warn.append("Exposure type dimension has no code / description column; credit_exposure_type left empty")
show(["<h3>Inputs and month plan</h3>",
      tbl(pd.DataFrame([("Analysis months", f"{MONTHS[0]} to {MONTHS[-1]} ({len(MONTHS)})"),
                        ("To build", f"{len(PLAN)}" + (f" (from {PLAN[0]})" if PLAN else "")),
                        ("PKG TM power ids (pkg_cust_dim)", f"{N_PK:,}"),
                        ("PD grades in dimension", f"{len(pdp)} (PD scale /{PD_SCALE:g})"),
                        ("Output folders", f"{PATH_CUST} | {PATH_PROD}"), ("Signature", SIG)],
                       columns=["item", "value"]))]
     + [f"<p class='warn'>{esc(w)}</p>" for w in warn] + [tbl(mt)])


In [ ]:
# ============================================================
# 3. Checks on the latest month (account link, revenue reconciliation, row growth)
# ============================================================
if not RUN_CHECKS:
    show(["<h3>Checks skipped (RUN_CHECKS = False)</h3>"])
else:
    mL = MONTHS[-1]
    tL, tLp, trL = BAL_TK[snap(mL)], BAL_TK[snap(ym_add(mL, -1))], REV_TK[snap(mL)]

    def keyset(tk, flag):
        return (bal.filter((F.col("time_key") == tk) & (F.col("acct_dim_key") > 0))
                   .select(F.col("acct_dim_key").cast("long").alias("acct_dim_key")).distinct().withColumn(flag, F.lit(1)))

    rvL = rev.filter(F.col("time_key") == trL)
    lk = (rvL.select(F.col("acct_dim_key").cast("long").alias("acct_dim_key"), F.col("pxv_ofs").cast("double").alias("pxv"),
                     clean_pwr("cust_pwr_id").alias("own"))
             .join(keyset(tL, "_s"), "acct_dim_key", "left").join(keyset(tLp, "_p"), "acct_dim_key", "left")
             .agg(F.count(F.lit(1)).alias("rows"), F.sum((F.col("acct_dim_key") > 0).cast("int")).alias("keyed"),
                  F.sum(F.col("_s").isNotNull().cast("int")).alias("same"), F.sum(F.col("_p").isNotNull().cast("int")).alias("prev"),
                  F.sum(F.col("own").isNotNull().cast("int")).alias("own"), F.sum("pxv").alias("pxv"),
                  F.sum(F.when(F.col("_s").isNotNull(), F.col("pxv"))).alias("pxv_same")).collect()[0])
    LINK_SAME = (lk["same"] / lk["keyed"]) if lk["keyed"] else 0.0
    LT = pd.DataFrame([
        ("Revenue rows / with an account key (> 0)", f"{lk['rows']:,} / {lk['keyed']:,} ({fpct(lk['keyed'] / lk['rows'] if lk['rows'] else None)})"),
        ("Account found in the SAME balance snapshot", fpct(LINK_SAME)),
        ("Account found in the PREVIOUS balance snapshot", fpct(lk["prev"] / lk["keyed"] if lk["keyed"] else None)),
        ("Revenue (pxv) on accounts found in the same snapshot", fpct(lk["pxv_same"] / lk["pxv"] if lk["pxv"] else None)),
        ("Rows with their own power id", fpct(lk["own"] / lk["rows"] if lk["rows"] else None))], columns=["check", "value"])

    rv_sys = (rvL.groupBy(F.col("subsystem_identifier_ofs").alias("system"))
                 .agg(F.count(F.lit(1)).alias("revenue rows"), F.sum(F.col("pxv_ofs").cast("double")).alias("pxv_ofs"),
                      F.sum(F.col("other_rev_ofs").cast("double")).alias("other_rev_ofs"),
                      F.sum(F.col("memo_rev_ofs").cast("double")).alias("memo_rev_ofs")).toPandas())
    bl_sys = (bal.filter(F.col("time_key") == tL).groupBy(F.col("data_src_cd").alias("system"))
                 .agg(F.count(F.lit(1)).alias("accounts"), F.sum(F.col("tot_pxv_rev").cast("double")).alias("tot_pxv_rev"),
                      F.sum(F.col("tm_pxv_rev").cast("double")).alias("tm_pxv_rev"),
                      F.sum(F.col("analyzed_chrg").cast("double")).alias("analyzed_chrg"),
                      F.sum(F.col("tot_bdf_billed").cast("double")).alias("tot_bdf_billed")).toPandas())
    RC = rv_sys.merge(bl_sys, on="system", how="outer").fillna(0)
    RC = pd.concat([RC, pd.DataFrame([{"system": "TOTAL", **{c: RC[c].sum() for c in RC.columns if c != "system"}}])])
    RC["revenue pxv / balance tot_pxv_rev"] = [fpct(a / b) if b else "" for a, b in zip(RC["pxv_ofs"], RC["tot_pxv_rev"])]
    for c in ("revenue rows", "accounts"):
        RC[c] = RC[c].map(lambda v: f"{int(v):,}")
    for c in ("pxv_ofs", "other_rev_ofs", "memo_rev_ofs", "tot_pxv_rev", "tm_pxv_rev", "analyzed_chrg", "tot_bdf_billed"):
        RC[c] = RC[c].map(fnum)

    gparts = []
    tg = REV_TK.get(snap(GROWTH_FROM))
    if tg and tg != trL:
        prodc = F.coalesce(F.trim(F.col("prod_desc_ofs")), F.lit("(none)"))
        gr = (rev.filter(F.col("time_key").isin(tg, trL)).groupBy("time_key", prodc.alias("product"))
                 .agg(F.count(F.lit(1)).alias("rows"), F.sum(F.col("pxv_ofs").cast("double")).alias("pxv")).toPandas())
        pv = gr.pivot_table(index="product", columns="time_key", values=["rows", "pxv"], aggfunc="sum").fillna(0)
        G = pd.DataFrame({"product": pv.index, "rows before": pv[("rows", tg)].values, "rows now": pv[("rows", trL)].values,
                          "pxv before": pv[("pxv", tg)].values, "pxv now": pv[("pxv", trL)].values})
        G["row change"] = G["rows now"] - G["rows before"]
        tot = G[["rows before", "rows now", "pxv before", "pxv now", "row change"]].sum()
        G = G.reindex(G["row change"].abs().sort_values(ascending=False).index).head(15)
        G = pd.concat([G, pd.DataFrame([{"product": "ALL PRODUCTS", **tot.to_dict()}])])
        for c in ("rows before", "rows now", "row change"):
            G[c] = G[c].map(lambda v: f"{int(v):+,}" if c == "row change" else f"{int(v):,}")
        for c in ("pxv before", "pxv now"):
            G[c] = G[c].map(fnum)
        gparts = [f"<h4>Row growth by product: analysis month {esc(GROWTH_FROM)} vs {esc(mL)} (15 largest changes)</h4>", tbl(G)]
    show([f"<h3>Checks — analysis month {esc(mL)} (balance snapshot {tL}, revenue snapshot {trL})</h3>",
          "<h4>Revenue to balance account link (acct_dim_key)</h4>", tbl(LT),
          f"<p class='note'>The build needs at least {fpct(MIN_ACCT_LINK)} in the same snapshot; acct_dim_key changes every "
          "month, so a low share for the previous snapshot is expected.</p>",
          "<h4>Revenue reconciliation by system (revenue fact vs balance table)</h4>", tbl(RC)] + gparts)
    if LINK_SAME < MIN_ACCT_LINK:
        raise RuntimeError(f"Only {LINK_SAME:.1%} of keyed revenue rows find their account in the same balance snapshot")


In [ ]:
# ============================================================
# 4. Build (customer fact staged once, then month by month)
# ============================================================
LOG_CSV = os.path.join(OUT_DIR, "tm_cust_month_runlog.csv")
CUST_COLS = (["customer_pwr_id", "in_ofs", "had_ofs_prev", "cust_gone", "in_cust_fact", "in_pkg_dim",
              "n_acct", "n_acct_open", "n_acct_new", "n_acct_closed_in_month", "n_acct_gone"]
             + list(BAL_SUMS) + ["ecr_rate_wavg", "ecr_rate_max", "n_fact_rows"]
             + [x for h in ("now", "1m", "3m", "12m") for x in (f"pd_grade_{h}", f"pd_pct_{h}")] + ["is_default"]
             + list(FACT_SUMS) + list(FACT_MAX) + ["iris_condition", "iris_direction", "iris_score", "credit_exposure_type"])
ZERO_COLS = ["in_ofs", "had_ofs_prev", "in_cust_fact", "in_pkg_dim", "n_acct", "n_acct_open", "n_acct_new",
             "n_acct_closed_in_month", "n_acct_gone"] + list(BAL_SUMS)

def close_enough(a, b):
    a, b = float(a or 0), float(b or 0)
    return abs(a - b) <= max(1e-6 * max(abs(a), abs(b)), 0.01)

def stage_fact(months):
    """Customer fact for the given analysis months, one row per power id, in one pass.
    Month-end dates hold two full loads and small reloads; only the latest full load of the date is used."""
    asof = {FACT_SNAP[m][0]: m for m in months}
    mo_map = F.create_map(*[x for a, m in asof.items() for x in (F.lit(a), F.lit(m))])
    tk_map = F.create_map(*[x for m in months for x in (F.lit(FACT_SNAP[m][0]), F.lit(FACT_SNAP[m][1]))])
    wv = Window.partitionBy("as_of_dt", "cust_dim_key").orderBy(F.col("time_key").desc())
    def pdc(c, tag):
        k = F.col(c).cast("long")
        return [PD_PCT[k].alias(f"pd_pct_{tag}"), PD_CD[k].alias(f"pd_grade_{tag}")]
    vals = [(F.col(s).cast("double") if s in fact.columns else F.lit(None).cast("double")).alias(d)
            for d, s in {**FACT_SUMS, **FACT_MAX}.items()]
    exc = (EX_MAP[F.col("cust_credit_expsr_type_dim_key").cast("long")] if EX_MAP is not None
           else F.lit(None).cast("string")).alias("credit_exposure_type")
    fx = (fact.filter(F.col("as_of_dt").isin(list(asof)))
              .filter(F.col("time_key") == tk_map[F.col("as_of_dt")])
              .withColumn("_rv", F.row_number().over(wv)).filter("_rv = 1")
              .select(mo_map[F.col("as_of_dt")].alias("month"), clean_pwr("cust_pwr_id").alias("customer_pwr_id"),
                      "cust_dim_key", *pdc("pd_key", "now"), *pdc("pd_cd_1_mnth_key", "1m"),
                      *pdc("pd_cd_3_mnth_key", "3m"), *pdc("pd_cd_12_mnth_key", "12m"),
                      F.col("iris_dim_key").cast("long").alias("iris_dim_key"), exc, *vals)
              .filter(F.col("customer_pwr_id").isNotNull())
              .join(F.broadcast(IRIS), "iris_dim_key", "left"))
    def worst(tag):   # grade of the customer's worst-PD row
        return (F.max(F.struct(F.coalesce(F.col(f"pd_pct_{tag}"), F.lit(-1.0)).alias("p"), F.col(f"pd_grade_{tag}").alias("g")))
                 .getField("g").alias(f"pd_grade_{tag}"))
    pick = F.max(F.struct(F.coalesce(F.col("dhe"), F.lit(0.0)).alias("a"), F.coalesce(F.col("gross_rev_12m"), F.lit(0.0)).alias("b"),
                          "iris_condition", "iris_direction", "iris_score", "credit_exposure_type")).alias("_pick")
    agg = (fx.groupBy("month", "customer_pwr_id")
             .agg(F.count(F.lit(1)).alias("n_fact_rows"),
                  *[x for h in ("now", "1m", "3m", "12m") for x in (worst(h), F.max(f"pd_pct_{h}").alias(f"pd_pct_{h}"))],
                  *[F.sum(d).alias(d) for d in FACT_SUMS], *[F.max(d).alias(d) for d in FACT_MAX], pick)
             .withColumn("is_default", F.coalesce((F.col("pd_pct_now") >= 0.999).cast("int"), F.lit(0)))
             .select("month", "customer_pwr_id", "n_fact_rows",
                     *[x for h in ("now", "1m", "3m", "12m") for x in (f"pd_grade_{h}", f"pd_pct_{h}")], "is_default",
                     *FACT_SUMS, *FACT_MAX, *[F.col(f"_pick.{c}").alias(c) for c in
                                              ("iris_condition", "iris_direction", "iris_score", "credit_exposure_type")]))
    # Plain overwrite of a fresh folder (no dynamic partition overwrite: there, retried / speculative tasks
    # write straight into the job folder and their files can survive, duplicating rows)
    fs_delete(STAGE)
    agg.write.mode("overwrite").partitionBy("month").parquet(STAGE)
    return check_stage()

def check_stage():
    """Staged customer fact must hold one row per (month, power id)."""
    st = spark.read.parquet(STAGE)
    r = (st.groupBy("month", "customer_pwr_id").count()
           .agg(F.count(F.lit(1)).alias("ids"), F.sum("count").alias("rows"),
                F.sum(F.when(F.col("count") > 1, 1).otherwise(0)).alias("dup")).collect()[0])
    if r["dup"]:
        raise RuntimeError(f"Customer fact staging has {r['dup']:,} (month, power id) pairs on 2+ rows "
                           f"({r['rows']:,} rows for {r['ids']:,} pairs)")
    return r["rows"]

def accounts(tk):
    """Balance rows of one snapshot, one per account key (latest seq_id), with _rn to count repeats."""
    w = Window.partitionBy(*KEY).orderBy(F.col("seq_id").desc())
    return (bal.filter(F.col("time_key") == tk)
               .select(*[F.coalesce(F.trim(F.col(c).cast("string")), F.lit("")).alias(c) for c in KEY], "seq_id",
                       F.col("acct_dim_key").cast("long").alias("acct_dim_key"), clean_pwr("cust_pwr_id").alias("pwr"),
                       F.col("curnt_cust_dim_key").cast("long").alias("ccdk"), F.col("cust_dim_key").cast("long").alias("cdk"),
                       (F.lit(True) if not LEAF_ONLY_AMOUNTS else
                        F.coalesce(F.upper(F.trim(F.col("hier_leaf_node").cast("string"))) != "N", F.lit(True))).alias("leaf"),
                       real_dt("acct_close_dt").alias("close_dt"), F.col("curnt_mnth_nbr").cast("int").alias("cmn"),
                       F.col("earnings_credit_rate").cast("double").alias("ecr_rate"),
                       *[F.col(s).cast("double").alias(d) for d, s in BAL_SUMS.items()])
               .filter(F.col("src_acct_no") != "")
               .withColumn("_rn", F.row_number().over(w)))

def key_map(df, key):
    """key -> power id from balance rows; key kept only when it points to one power id."""
    return (df.filter(F.col(key) > 0).groupBy(F.col(key).alias("_k"))
              .agg(F.countDistinct("pwr").alias("_np"), F.max("pwr").alias("_pm"))
              .select("_k", F.lit(1).alias("_inb"), F.when(F.col("_np") == 1, F.col("_pm")).alias("pwr_link")))

OWN_ONLY = "own power id only (partial)"
_CDM = {"done": False, "map": None, "note": ""}
def cust_dim_map():
    """cust_dim_key -> power id from the customer dimension; None when the table or columns are not there."""
    if not _CDM["done"]:
        _CDM["done"] = True
        try:
            d = spark.table(CUST_DIM)
            pc = next((c for c in ("cust_pwr_id", "customer_pwr_id", "pwr_id") if c in d.columns), None)
            if "cust_dim_key" in d.columns and pc:
                _CDM["map"] = key_map(d.select(F.col("cust_dim_key").cast("long").alias("dk"), clean_pwr(pc).alias("pwr")),
                                      "dk").persist(StorageLevel.MEMORY_AND_DISK)
            else:
                _CDM["note"] = f"{CUST_DIM} lacks cust_dim_key or a power id column"
        except Exception as e:
            _CDM["note"] = f"cannot read {CUST_DIM}: {str(e).splitlines()[0][:150]}"
    return _CDM["map"]

def loads_in_month(table, ym):
    """{time_key: rows} for one snapshot month (diagnostics)."""
    r = (spark.table(table).filter(F.col("time_key").between(ym * 100 + 1, ym * 100 + 31))
              .groupBy("time_key").count().toPandas())
    return {int(t): int(c) for t, c in zip(r["time_key"], r["count"])}

def prev_first_seen(m):
    """(power id, product) -> first month seen, for all months before m."""
    p = f"{STATE}/month={ym_add(m, -1)}"
    if fs_exists(p + "/_SUCCESS"):
        return spark.read.parquet(p)
    paths = [f"{PATH_PROD}/month={x}" for x in MONTHS if x < m]   # fallback: rebuild from earlier product months
    return (spark.read.parquet(*paths).groupBy("customer_pwr_id", "tm_product")
                 .agg(F.min("first_month").alias("first_month")))

def build_month(m, prev_id):
    t0 = time.time()
    t, tp, tr = BAL_TK[snap(m)], BAL_TK[snap(ym_add(m, -1))], REV_TK[snap(m)]
    ms = _dt.date(int(m[:4]), int(m[5:7]), 1)
    me = (ms + _dt.timedelta(days=32)).replace(day=1) - _dt.timedelta(days=1)

    # ---- balance: accounts this month and last month ----
    raw = accounts(t).persist(StorageLevel.MEMORY_AND_DISK)
    s0 = raw.agg(F.count(F.lit(1)).alias("rows"), F.sum((F.col("_rn") > 1).cast("int")).alias("rep"),
                 F.avg((F.col("cmn") == int(m[5:7])).cast("int")).alias("cmn_ok")).collect()[0]
    if s0["rows"] == 0:
        raise RuntimeError(f"{m}: balance snapshot {t} is empty")
    if (s0["cmn_ok"] or 0) < 0.95:
        raise RuntimeError(f"{m}: only {s0['cmn_ok']:.1%} of balance rows in snapshot {t} carry curnt_mnth_nbr "
                           f"{int(m[5:7])}; the month shift does not hold")
    cur = raw.filter("_rn = 1").drop("_rn")
    prv = (accounts(tp).filter("_rn = 1").select(*KEY, F.col("pwr").alias("pwr_prev"))
                       .withColumn("_in_prev", F.lit(1)).persist(StorageLevel.MEMORY_AND_DISK))
    cj = cur.join(prv.select(*KEY, "_in_prev"), KEY, "left")
    pos_inv = F.when(F.col("leaf"), F.greatest(F.col("inv_bal"), F.lit(0.0)))
    cb = (cj.filter(F.col("pwr").isNotNull()).groupBy(F.col("pwr").alias("customer_pwr_id"))
            .agg(F.count(F.lit(1)).alias("n_acct"), F.sum(F.col("close_dt").isNull().cast("int")).alias("n_acct_open"),
                 F.sum(F.col("_in_prev").isNull().cast("int")).alias("n_acct_new"),
                 F.sum(F.coalesce(F.col("close_dt").between(F.lit(ms), F.lit(me)).cast("int"), F.lit(0))).alias("n_acct_closed_in_month"),
                 *[F.sum(F.when(F.col("leaf"), F.col(d))).alias(d) for d in BAL_SUMS],
                 (F.sum(F.when(F.col("ecr_rate") > 0, F.col("ecr_rate") * pos_inv))
                  / F.sum(F.when(F.col("ecr_rate") > 0, pos_inv))).alias("ecr_rate_wavg"),
                 F.max(F.when(F.col("leaf"), F.col("ecr_rate"))).alias("ecr_rate_max"))
            .withColumn("in_ofs", F.lit(1)))
    gone = (prv.join(cur.select(*KEY).withColumn("_in_cur", F.lit(1)), KEY, "left")
               .filter(F.col("_in_cur").isNull() & F.col("pwr_prev").isNotNull())
               .groupBy(F.col("pwr_prev").alias("customer_pwr_id")).agg(F.count(F.lit(1)).alias("n_acct_gone")))
    prevc = (prv.filter(F.col("pwr_prev").isNotNull()).select(F.col("pwr_prev").alias("customer_pwr_id")).distinct()
                .withColumn("had_ofs_prev", F.lit(1)))

    # ---- customer fact (staged) and universe ----
    fm = spark.read.parquet(f"{STAGE}/month={m}").withColumn("in_cust_fact", F.lit(1))
    U = (cb.select("customer_pwr_id").union(prevc.select("customer_pwr_id"))
           .union(fm.join(pk, "customer_pwr_id", "left_semi").select("customer_pwr_id")).distinct())
    cust = (U.join(cb, "customer_pwr_id", "left").join(prevc, "customer_pwr_id", "left")
             .join(gone, "customer_pwr_id", "left").join(fm, "customer_pwr_id", "left")
             .join(pk.withColumn("in_pkg_dim", F.lit(1)), "customer_pwr_id", "left")
             .fillna(0, subset=ZERO_COLS)
             .withColumn("cust_gone", ((F.col("had_ofs_prev") == 1) & (F.col("in_ofs") == 0)).cast("int"))
             .select(*CUST_COLS).persist(StorageLevel.MEMORY_AND_DISK))

    rc = cust.agg(F.count(F.lit(1)).alias("rows"), F.countDistinct("customer_pwr_id").alias("ids"),
                  F.sum("n_acct").alias("n_acct"), F.sum("inv_bal").alias("inv_bal"), F.sum("n_acct_gone").alias("gone"),
                  F.sum("dhe").alias("dhe"), F.sum("in_ofs").alias("in_ofs"), F.sum("cust_gone").alias("cust_gone"),
                  F.sum("in_cust_fact").alias("in_fact"), F.sum("billed").alias("billed"),
                  F.sum(F.col("in_ofs") * F.col("in_cust_fact")).alias("ofs_in_fact")).collect()[0]
    if rc["in_ofs"] and rc["ofs_in_fact"] / rc["in_ofs"] < 0.80:
        raise RuntimeError(f"{m}: only {rc['ofs_in_fact'] / rc['in_ofs']:.1%} of TM analysis customers found in the "
                           f"customer fact ({FACT_SNAP[m][0]}); check the customer fact snapshot rule")
    ea = (cur.filter(F.col("pwr").isNotNull())
             .agg(F.count(F.lit(1)).alias("n"), F.sum(F.when(F.col("leaf"), F.col("inv_bal"))).alias("inv")).collect()[0])
    eg = gone.agg(F.sum("n_acct_gone")).collect()[0][0] or 0
    ed = fm.join(U, "customer_pwr_id", "left_semi").agg(F.sum("dhe")).collect()[0][0]
    na = cur.agg(F.count(F.lit(1)).alias("n"), F.sum(F.col("pwr").isNotNull().cast("int")).alias("p")).collect()[0]
    bad = [x for x, ok in (("one row per power id", rc["rows"] == rc["ids"]), ("accounts", rc["n_acct"] == ea["n"]),
                           ("investable balance", close_enough(rc["inv_bal"], ea["inv"])), ("accounts gone", rc["gone"] == eg),
                           ("direct hard exposure", close_enough(rc["dhe"], ed))) if not ok]
    if bad:
        def dups(df):
            return df.groupBy("customer_pwr_id").count().filter("count > 1").count()
        diag = {"rows": rc["rows"], "distinct power ids": rc["ids"], "accounts table vs source": (rc["n_acct"], ea["n"]),
                "investable table vs source": (rc["inv_bal"], ea["inv"]), "gone table vs source": (rc["gone"], eg),
                "dhe table vs source": (rc["dhe"], ed), "repeated ids: customer fact": dups(fm),
                "repeated ids: TM accounts": dups(cb), "repeated ids: previous month": dups(prevc),
                "repeated ids: gone": dups(gone), "repeated ids: pkg_cust_dim": dups(pk)}
        raise RuntimeError(f"{m}: customer table does not reconcile: {bad}; {diag}")

    # ---- revenue: link to customers, products ----
    prod_c = F.trim(F.col("prod_desc_ofs").cast("string"))
    rv = (rev.filter(F.col("time_key") == tr)
             .select(F.col("acct_dim_key").cast("long").alias("acct_dim_key"),
                     F.col("curnt_cust_dim_key").cast("long").alias("ccdk"), F.col("cust_dim_key").cast("long").alias("cdk"),
                     clean_pwr("cust_pwr_id").alias("pwr_own"),
                     F.when(prod_c.isNull() | (prod_c == ""), F.lit("(none)")).otherwise(prod_c).alias("tm_product"),
                     F.trim(F.col("tm_service_cd_ofs").cast("string")).alias("svc"),
                     F.col("volume_ofs").cast("double").alias("vol"), F.col("pxv_ofs").cast("double").alias("pxv"))
             .persist(StorageLevel.MEMORY_AND_DISK))
    if rv.limit(1).count() == 0:
        raise RuntimeError(f"{m}: revenue snapshot {tr} is empty")
    # Link ladder, tried in order; the first method that links >= MIN_ACCT_LINK of the revenue rows carrying an
    # account key is used. Only rows with an account key are linked this way, so the row population is the same
    # whatever the method; rows without one keep their own power id.
    keyed = rv.filter(F.col("acct_dim_key") > 0)
    def rate(mp, key):
        r = (keyed.join(mp, F.col(key) == F.col("_k"), "left")
                  .agg(F.count(F.lit(1)).alias("n"), F.sum(F.col("_inb").isNotNull().cast("int")).alias("hit")).collect()[0])
        return (r["hit"] / r["n"]) if r["n"] else 0.0
    tn = BAL_TK.get(snap(ym_add(m, 1)))
    ladder = [("account key, same snapshot", "acct_dim_key", lambda: key_map(cur, "acct_dim_key")),
              ("account key, previous snapshot", "acct_dim_key", lambda: key_map(accounts(tp).filter("_rn = 1"), "acct_dim_key"))]
    if tn:
        ladder.append(("account key, next snapshot", "acct_dim_key", lambda: key_map(accounts(tn).filter("_rn = 1"), "acct_dim_key")))
    ladder += [("current customer key, same snapshot", "ccdk", lambda: key_map(cur, "ccdk")),
               ("customer key, same snapshot", "cdk", lambda: key_map(cur, "cdk")),
               ("customer dimension, customer key", "cdk", cust_dim_map),
               ("customer dimension, current customer key", "ccdk", cust_dim_map)]
    rates, chosen = {}, None
    for name, key, make in ladder:
        mp = make()
        if mp is None:          # customer dimension not usable
            continue
        rates[name] = rate(mp, key)
        if rates[name] >= MIN_ACCT_LINK:
            chosen = (name, key, mp)
            break
    if chosen is None:
        own = keyed.agg(F.avg(F.col("pwr_own").isNotNull().cast("int"))).collect()[0][0] or 0.0
        if not ALLOW_OWN_ID_ONLY:
            diag = {"link rates": {k: round(v, 4) for k, v in rates.items()}, "own power id share": round(own, 4),
                    "customer dimension": _CDM["note"] or "used",
                    "balance loads in snapshot month": loads_in_month(OFS_BAL, snap(m)),
                    "revenue loads in snapshot month": loads_in_month(OFS_REV, snap(m))}
            raise RuntimeError(f"{m}: no method links {MIN_ACCT_LINK:.0%} of keyed revenue rows to a customer; {diag}")
        rates[OWN_ONLY] = own
        chosen = (OWN_ONLY, None, None)
    method, key, mp = chosen
    if key is None:             # partial month: revenue rows keep only their own power id
        rj = (rv.withColumn("_inb", F.lit(None).cast("int")).withColumn("pwr_link", F.lit(None).cast("string"))
                .withColumn("customer_pwr_id", F.col("pwr_own")).persist(StorageLevel.MEMORY_AND_DISK))
    else:
        rj = (rv.join(mp, F.col(key) == F.col("_k"), "left").drop("_k")
                .withColumn("_inb", F.when(F.col("acct_dim_key") > 0, F.col("_inb")))
                .withColumn("pwr_link", F.when(F.col("acct_dim_key") > 0, F.col("pwr_link")))
                .withColumn("customer_pwr_id", F.coalesce("pwr_link", "pwr_own"))
                .persist(StorageLevel.MEMORY_AND_DISK))
    lr = rj.agg(F.count(F.lit(1)).alias("rows"), F.sum((F.col("acct_dim_key") > 0).cast("int")).alias("keyed"),
                F.sum(F.col("_inb").isNotNull().cast("int")).alias("inb"),
                F.sum(F.col("pwr_link").isNotNull().cast("int")).alias("via_acct"),
                F.sum((F.col("pwr_link").isNull() & F.col("pwr_own").isNotNull()).cast("int")).alias("via_own"),
                F.sum("pxv").alias("pxv"), F.sum(F.when(F.col("customer_pwr_id").isNotNull(), F.col("pxv"))).alias("pxv_linked")
                ).collect()[0]
    link = rates["account key, same snapshot"]
    prod = (rj.filter(F.col("customer_pwr_id").isNotNull()).groupBy("customer_pwr_id", "tm_product")
              .agg(F.countDistinct("svc").alias("n_service_cd"),
                   F.countDistinct(F.when(F.col("acct_dim_key") > 0, F.col("acct_dim_key"))).alias("n_acct"),
                   F.count(F.lit(1)).alias("n_rows"), F.sum((F.col("vol") > 0).cast("int")).alias("n_rows_volume"),
                   F.sum("pxv").alias("pxv")))
    if m == MONTHS[0]:
        prod = prod.withColumn("first_month", F.lit(m))
        seen = prod.select("customer_pwr_id", "tm_product", "first_month")
    else:
        st = prev_first_seen(m)
        prod = (prod.join(st.withColumnRenamed("first_month", "_fm"), ["customer_pwr_id", "tm_product"], "left")
                    .withColumn("first_month", F.coalesce("_fm", F.lit(m))).drop("_fm"))
        seen = (st.unionByName(prod.select("customer_pwr_id", "tm_product", "first_month"))
                  .groupBy("customer_pwr_id", "tm_product").agg(F.min("first_month").alias("first_month")))
    prod = prod.select("customer_pwr_id", "tm_product", "n_service_cd", "n_acct", "n_rows", "n_rows_volume", "pxv",
                       "first_month").persist(StorageLevel.MEMORY_AND_DISK)
    rp = prod.agg(F.count(F.lit(1)).alias("rows"), F.sum("pxv").alias("pxv"), F.sum("n_rows").alias("n_rows"),
                  F.sum((F.col("first_month") == m).cast("int")).alias("new_pairs")).collect()[0]
    nl = rj.filter(F.col("customer_pwr_id").isNotNull()).count()
    if rp["n_rows"] != nl or not close_enough(rp["pxv"], lr["pxv_linked"]):
        raise RuntimeError(f"{m}: product table does not reconcile (rows {rp['n_rows']} vs {nl}, "
                           f"pxv {rp['pxv']} vs {lr['pxv_linked']})")

    # ---- write, then markers (a month counts as done only with both markers) ----
    for p in (f"{PATH_CUST}/month={m}", f"{PATH_PROD}/month={m}"):
        fs_delete(p + "/_PKG_MARKER.json")
    cust.repartition(16).write.mode("overwrite").parquet(f"{PATH_CUST}/month={m}")
    prod.repartition(32).write.mode("overwrite").parquet(f"{PATH_PROD}/month={m}")
    seen.repartition(32).write.mode("overwrite").parquet(f"{STATE}/month={m}")
    bid = uuid.uuid4().hex[:12]
    mk = json.dumps({"sig": SIG, "build_id": bid, "prev": prev_id, "month": m, "built": _dt.datetime.now().isoformat(timespec="seconds")})
    fs_write(f"{PATH_CUST}/month={m}/_PKG_MARKER.json", mk)
    fs_write(f"{PATH_PROD}/month={m}/_PKG_MARKER.json", mk)
    old = ym_add(m, -2)
    if old >= MONTHS[0]:
        fs_delete(f"{STATE}/month={old}")       # keep the last two first-seen states
    for d in (raw, prv, cust, rv, rj, prod):
        d.unpersist()
    return bid, {"month": m, "balance snapshot": t, "fact as_of_dt": FACT_SNAP[m][0], "balance rows": s0["rows"],
                 "repeated account keys": s0["rep"], "accounts with power id": na["p"] / na["n"],
                 "customers": rc["rows"], "in OFS": rc["in_ofs"], "gone": rc["cust_gone"], "in customer fact": rc["in_fact"],
                 "accounts": rc["n_acct"], "investable balance": rc["inv_bal"], "billed": rc["billed"],
                 "direct hard exposure": rc["dhe"], "revenue rows": lr["rows"], "account link": link,
                 "link method": method, "link used": rates[method],
                 "link rates tried": "" if method == "account key, same snapshot" else
                                     "; ".join(f"{k} {v:.1%}" for k, v in rates.items()),
                 "rows via account": lr["via_acct"] / lr["rows"], "rows via own id": lr["via_own"] / lr["rows"],
                 "pxv linked": (lr["pxv_linked"] / lr["pxv"]) if lr["pxv"] else None, "product rows": rp["rows"],
                 "new product pairs": rp["new_pairs"], "pxv": rp["pxv"], "seconds": round(time.time() - t0)}

def log_view(rows):
    if not rows:
        return "<p class='note'>(nothing built yet)</p>"
    v = pd.DataFrame(rows)
    for c in ("accounts with power id", "account link", "link used", "rows via account", "rows via own id", "pxv linked"):
        v[c] = v[c].map(fpct)
    for c in ("investable balance", "billed", "direct hard exposure", "pxv"):
        v[c] = v[c].map(fnum)
    for c in ("balance rows", "repeated account keys", "customers", "in OFS", "gone", "in customer fact", "accounts",
              "revenue rows", "product rows", "new product pairs"):
        v[c] = v[c].map(lambda x: f"{int(x):,}")
    return tbl(v)

PLAN, status = plan_months()
done_now = []
if not PLAN:
    show(["<h3>Build</h3><p>All months are up to date; nothing to build.</p>"])
else:
    prev_id = dict((m, b) for m, _, b in status).get(ym_add(PLAN[0], -1))
    hd = display(HTML(CSS + f"<div class='cx'><h3>Build</h3><p>Staging customer fact for {len(PLAN)} months ...</p></div>"),
                 display_id=True)
    t_all = time.time()
    n_staged = stage_fact(PLAN)
    hd.update(HTML(CSS + f"<div class='cx'><h3>Build</h3><p>Customer fact staged: {n_staged:,} rows, one per month and "
                         "power id. Building months ...</p></div>"))
    try:
        for i, m in enumerate(PLAN):
            prev_id, row = build_month(m, prev_id)
            done_now.append(row)
            el = time.time() - t_all
            hd.update(HTML(CSS + f"<div class='cx'><h3>Build — {i + 1} of {len(PLAN)} months done, "
                                 f"{el / 60:.0f} min so far</h3>" + log_view(done_now) + "</div>"))
    finally:
        if done_now:
            old = pd.read_csv(LOG_CSV) if os.path.exists(LOG_CSV) else pd.DataFrame()
            new = pd.DataFrame(done_now)
            if len(old):
                old = old[~old["month"].isin(new["month"])]
            pd.concat([old, new]).sort_values("month").to_csv(LOG_CSV, index=False)
    hd.update(HTML(CSS + f"<div class='cx'><h3>Build — {len(PLAN)} months built in {(time.time() - t_all) / 60:.0f} min</h3>"
                   + log_view(done_now) + f"<p class='note'>Run log: {esc(os.path.abspath(LOG_CSV))}</p></div>"))


In [ ]:
# ============================================================
# 5. Register tables, monthly summary, column dictionary
# ============================================================
PLAN, status = plan_months(use_rebuild=False)
READY = [m for m, s, _ in status if s == "done"]
if not READY:
    raise RuntimeError("No month is complete; run cell 4 first")

def register(table, path, months):
    full = f"{DB}.{table}"
    sch = spark.read.parquet(f"{path}/month={months[-1]}").schema
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in sch.fields)
    if spark.sql(f"SHOW TABLES IN {DB} LIKE '{table}'").count() > 0:
        d = spark.sql(f"DESCRIBE TABLE EXTENDED {full}").toPandas()
        loc = d.loc[d["col_name"].astype(str).str.strip() == "Location", "data_type"].astype(str).tolist()
        if not loc or loc[0].rstrip("/") != path.rstrip("/"):
            raise RuntimeError(f"{full} already exists at {loc}; not replacing it")
        spark.sql(f"DROP TABLE {full}")          # external table: files stay
    spark.sql(f"CREATE TABLE {full} ({cols}, month STRING) USING parquet PARTITIONED BY (month) LOCATION '{path}'")
    for m in months:
        spark.sql(f"ALTER TABLE {full} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{path}/month={m}'")
    return full

fc, fp = register(T_CUST, PATH_CUST, READY), register(T_PROD, PATH_PROD, READY)
cnt_c = spark.table(fc).groupBy("month").count().toPandas().set_index("month")["count"]
cnt_p = spark.table(fp).groupBy("month").count().toPandas().set_index("month")["count"]
LOG = pd.read_csv(LOG_CSV) if os.path.exists(LOG_CSV) else pd.DataFrame(columns=["month"])
LOG = LOG[LOG["month"].isin(READY)]
SUM = pd.DataFrame({"month": READY})
SUM["customer rows (table)"] = SUM["month"].map(lambda m: f"{int(cnt_c.get(m, 0)):,}")
SUM["product rows (table)"] = SUM["month"].map(lambda m: f"{int(cnt_p.get(m, 0)):,}")
if len(LOG):
    if "link method" not in LOG.columns:
        LOG["link method"] = None
    LOG["link method"] = LOG["link method"].fillna("account key, same snapshot")
    L2 = LOG.set_index("month")
    for c, f in (("in OFS", lambda x: f"{int(x):,}"), ("gone", lambda x: f"{int(x):,}"),
                 ("in customer fact", lambda x: f"{int(x):,}"), ("accounts", lambda x: f"{int(x):,}"),
                 ("investable balance", fnum), ("billed", fnum), ("direct hard exposure", fnum),
                 ("account link", fpct), ("link method", str), ("pxv linked", fpct), ("pxv", fnum)):
        SUM[c] = SUM["month"].map(lambda m: f(L2.loc[m, c]) if m in L2.index and pd.notna(L2.loc[m, c]) else "")
    PARTIAL = [m for m in READY if m in L2.index and L2.loc[m, "link method"] == OWN_ONLY]
    FALLBACK = [m for m in READY if m in L2.index and L2.loc[m, "link method"] not in ("account key, same snapshot", OWN_ONLY)]
    dser = pd.Series({m: float(L2.loc[m, "direct hard exposure"]) for m in READY if m in L2.index})
    JUMPS = []
    for i in range(1, len(dser) - 1):
        a, b, c = dser.iloc[i - 1], dser.iloc[i], dser.iloc[i + 1]
        if a > 0 and c > 0 and (min(b / a, b / c) > DHE_JUMP or max(b / a, b / c) < 1 / DHE_JUMP):
            JUMPS.append(f"{dser.index[i]} ({fnum(b)} vs {fnum(a)} / {fnum(c)})")
    bad = [m for m in READY if m in L2.index and (int(cnt_c.get(m, -1)) != int(L2.loc[m, "customers"])
                                                  or int(cnt_p.get(m, -1)) != int(L2.loc[m, "product rows"]))]
else:
    bad, PARTIAL, FALLBACK, JUMPS = [], [], [], []

DICT = pd.DataFrame([
    ("both", "month", "Analysis month YYYY-MM (partition); same meaning as PKG months (OFS snapshot of the following month)"),
    ("both", "customer_pwr_id", "Power id, 10 digits"),
    (T_CUST, "in_ofs / had_ofs_prev", "1 = has TM analysis accounts this month / had them last month"),
    (T_CUST, "cust_gone", "1 = had TM analysis accounts last month, none this month"),
    (T_CUST, "in_cust_fact / in_pkg_dim", "1 = found in the customer fact this month / in pkg_cust_dim"),
    (T_CUST, "n_acct, n_acct_open", "TM analysis accounts (bank + appl_id + src_acct_no); open = no real close date"),
    (T_CUST, "n_acct_new / n_acct_gone", "Accounts not in last month's snapshot / last month's accounts missing now (counted under last month's power id)"),
    (T_CUST, "n_acct_closed_in_month", "Accounts whose acct_close_dt falls in the month (CAA accounts only carry close dates)"),
    (T_CUST, "inv_bal, avg_bal", "Sum of avg_investable_bal / cor_tda_avg_bal over the customer's leaf accounts (all OFS amounts exclude parent rows of the billing hierarchy, which repeat their children)"),
    (T_CUST, "analyzed_chrg, waived, eca_used, billed", "Sum of analyzed_chrg, waived_results (negative), eca_used, tot_bdf_billed. Billed is lumpy (periodic billing spikes several times a month's level): use a 12-month rolling sum"),
    (T_CUST, "ecr_amt, ecr_rate_wavg, ecr_rate_max", "Earnings credit amount; rate (percent) weighted by positive investable balance over accounts with a rate; highest rate"),
    (T_CUST, "ftp_value", "Sum of cor_tda_ftp_on_avg_bal: funds-transfer-pricing value of the balances"),
    (T_CUST, "OFS counts and sums", "0 when the customer has no TM analysis accounts this month"),
    (T_CUST, "n_fact_rows", "Customer fact rows (cust_dim_key) behind the power id; amounts are summed over them"),
    (T_CUST, "pd_grade_*, pd_pct_*", "PD grade code and PD (fraction, 1.0 = defaulted) now / 1 / 3 / 12 months ago, worst row; empty = not rated"),
    (T_CUST, "is_default", "1 = current PD is 100% (defaulted grade)"),
    (T_CUST, "dhe, wrre, mre, soft_exposure, watchlist_exposure", "Sums of dhe, cust_wrre, cust_mre, cust_ste, watchlist_exposure"),
    (T_CUST, "past_due_days", "Largest cust_past_due_days"),
    (T_CUST, "*_rev_12m, ep_12m", "12-month gross / credit / non-credit / TM revenue and economic profit (ep_amt_12_mnth)"),
    (T_CUST, "iris_*, credit_exposure_type", "Industry risk condition, direction, score and exposure type of the row with the largest exposure"),
    (T_PROD, "tm_product", "prod_desc_ofs of the revenue fact"),
    (T_PROD, "n_service_cd, n_acct, n_rows", "Distinct service codes, accounts and revenue rows"),
    (T_PROD, "n_rows_volume", "Revenue rows with volume_ofs > 0 (volumes have different units per service, so they are not summed)"),
    (T_PROD, "pxv", "Sum of pxv_ofs (price x volume)"),
    (T_PROD, "first_month", f"First analysis month the customer used the product, counted from {MONTHS[0]}"),
    (T_PROD, "partial months", "Revenue linked only through the rows' own power id (no key links that month's revenue load): "
                               + (", ".join(PARTIAL) or "none") + ". Product holdings there are incomplete; do not read drops in those months as product loss."),
], columns=["table", "column", "definition"])
DICT.to_csv(os.path.join(OUT_DIR, "tm_cust_month_dictionary.csv"), index=False)

show([f"<h3>Registered {esc(fc)} and {esc(fp)} — {len(READY)} months, {READY[0]} to {READY[-1]}</h3>"]
     + ([f"<p class='warn'>Row counts differ from the run log for: {esc(', '.join(bad))}</p>"] if bad else [])
     + ([f"<p class='warn'>{len(PLAN)} month(s) still to build (from {PLAN[0]}); rerun cell 4.</p>"] if PLAN else [])
     + ([f"<p class='warn'>Partial product months (own power id only): {esc(', '.join(PARTIAL))}</p>"] if PARTIAL else [])
     + ([f"<p class='note'>Months linked by a fallback key: {esc(', '.join(FALLBACK))}</p>"] if FALLBACK else [])
     + ([f"<p class='warn'>DHE jumps against both neighbouring months: {esc('; '.join(JUMPS))}</p>"] if JUMPS else
        ["<p class='note'>DHE check: no month moves more than " + f"{DHE_JUMP:g}x against both neighbours.</p>"])
     + [tbl(SUM), "<p class='note'>In Impala: INVALIDATE METADATA " + esc(fc) + "; INVALIDATE METADATA " + esc(fp) + ";</p>",
        "<h4>Columns</h4>", tbl(DICT)])
